# FE 12 · Day-ahead 15-min features, v1 → `fe_dayahead_15min_v1`
Feature table for the **first model: day-ahead, 15-minute resolution**, both plants, **Jan 2024 → now**.

| | Agreed |
|---|---|
| Forecast origins | 09:00 and 15:00 Gulf time each day |
| Targets | every 15 min for the next 24 h (96 per origin) |
| Target | PV1 = actual MW · PV2 = `Pv2_AVL` (available power) |
| Night rows | **kept**, flagged with `is_day`, so two candidate models (with / without night) can be trained |
| Outages / gaps | kept, but `label_ok = false` → excluded from training and scoring |
| Broken Reuniwatt runs | detected and **not used** (e.g. 3 Jul 2026) |

**Feature groups**
| Prefix | Group | Known at |
|---|---|---|
| `da_`, `wa_`, `f_` | Reuniwatt forecast for the target (day-ahead run; week-ahead as fallback; `f_` = best available) | origin (run issued ≥ 15 min before) |
| `s_` | Plant sensors: conditions now and over the last 15–60 min, yesterday's haze | up to the origin |
| `c_` | Combined: Reuniwatt's recent error vs sensors and target; forecast adjusted by it | up to the origin |
| (none) | Computed: sun position, clear-sky, calendar, lead time | always |
| `label_` | The answer: target at the target time | — |

Run cell by cell. Steps 1–2 read ~2.75 years of Bronze and take the longest. Every step saves its result, so you can resume from any step.

## Step 0 · Settings

In [ ]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema")
dbutils.widgets.text("start_date", "2024-01-23", "First origin date")
dbutils.widgets.text("end_date", "", "Last origin date (blank = yesterday)")

In [ ]:
%run ./00_fe_common

In [ ]:
from datetime import date as _date, timedelta as _td

START = dbutils.widgets.get("start_date").strip()
END = dbutils.widgets.get("end_date").strip() or str(_date.today() - _td(days=1))
DATA_START = str(_date.fromisoformat(START) - _td(days=3))  # history needed for "last 24 h" / "yesterday" features
RES = 15                      # minutes
ORIGIN_HOURS_GULF = [9, 15]
WINDOW_MIN = 24 * 60
LATENCY_MIN = 15
MAX_RUN_AGE = {"dayahead": 360, "weekahead": 1440}
PREFIX = {"dayahead": "da", "weekahead": "wa"}
IRR_POWER = ["ghi", "dni", "dhi", "bhi", "gti_poa", "power_mw", "power_p10_mw", "power_p90_mw"]
WEATHER = ["temp_c", "dewpoint_c", "humidity_pct", "cloud_opacity", "clearness_index", "pressure_hpa", "precip_rate_mmh"]
print(f"Origins {START} → {END}; data from {DATA_START}; resolution {RES} min")

## Step 1 · Plant sensors + target (5-min) → `fe_sensor_target_5min`
Cleaning:
- Irradiance below −20 or above 1,500 → blank (−20 to 0 → 0).
- Temperature ≤ 0 or > 60 °C → blank.
- Humidity ≤ 0 or > 100% → blank.
- `Pv2_AVL` outside 0–2,000 → blank.
- MW: corrupt values removed and the Mar 2026 column swap fixed.

Converted Gulf → UTC and averaged to 5 min. Status per period: `night` / `missing` / `suspected_outage` / `ok`.

In [ ]:
st = write(build_sensor_target_5min(DATA_START, END), "fe_sensor_target_5min", partition_by=("site",))
display(st.groupBy("site", "status").count().orderBy("site", "status"))
display(st.where("is_day").groupBy("site", F.year("period_end").alias("year")).agg(
    *[F.round(100 * F.avg(F.col(k).isNotNull().cast("int")), 1).alias(f"{k}_pct") for k in ["m_ghi", "m_gti_poa", "m_temp_c", "m_humidity_pct", "target_mw"]],
    F.round(F.max("target_mw")).alias("target_max")).orderBy("site", "year"))

✅ **What you should see:**
- Mostly `night` and `ok`, a few `missing` / `suspected_outage`.
- Daytime coverage per year: `m_ghi`, `m_temp_c`, `m_humidity_pct` high every year.
- PV2 `m_gti_poa` only from **2026** (sensor started Jan 2026); PV2 `target_mw` from mid-2023 (`Pv2_AVL`).
- `target_max` ~850–950 (pv1) and ~1,640 (pv2).

## Step 2 · Reuniwatt day-ahead + week-ahead runs (5-min) → `fe_reuni_da_wa_5min`
From Bronze: hourly day-ahead runs, 3-hourly week-ahead runs (first 2 days only, used as fallback), power converted to MW.

In [ ]:
fc5 = write(reuniwatt_from_bronze(DATA_START, END, horizons=["dayahead", "weekahead"], max_lead_min=2 * 1440),
            "fe_reuni_da_wa_5min", partition_by=("horizon",))
display(fc5.groupBy("site", "horizon", F.year("reference_time").alias("year")).agg(
    F.countDistinct("reference_time").alias("runs"), F.countDistinct(F.to_date("reference_time")).alias("days"),
    F.round(F.max("power_mw")).alias("max_power_mw")).orderBy("site", "horizon", "year"))

✅ **What you should see:**
- The unit line printed **kW** (× 0.001).
- Per full year: ~**24 runs/day** for day-ahead and ~**8/day** for week-ahead.
- `max_power_mw` ~850–950 (pv1), ~1,600 (pv2).

## Step 3 · Flag broken Reuniwatt runs
A run is **broken** if, when the sun is high (clear-sky GHI > 600), more than half its GHI values are below 20% of clear-sky, i.e. it forecasts darkness at midday. Broken runs are not used.

In [ ]:
br = write(flag_broken_runs(fc5), "fe_reuni_run_flags")
display(br.groupBy("site", "horizon").agg(F.count("*").alias("runs"), F.sum(F.col("run_broken").cast("int")).alias("broken_runs")))
display(br.where("run_broken").groupBy("site", "horizon", F.to_date("reference_time").alias("day")).count().orderBy("day").limit(50))

✅ **What you should see:** a small number of broken runs (well under 1%), including runs on or just before **3 Jul 2026**. Hundreds of broken runs on normal days would mean the rule is too strict; tell Claude.

## Step 4 · Resample to 15 min
- Forecasts: average the three 5-min values in each 15-min period (broken runs removed) → `fe_reuni_da_wa_15min`.
- Target: average over the period; `label_ok` only if all three 5-min periods are usable.

In [ ]:
RUN_KEY = ["site", "horizon", "reference_time"]
fc15 = (
    fc5.join(br.select(*RUN_KEY, "run_broken"), RUN_KEY).where(~F.col("run_broken"))
    .withColumn("period_end", ceil_to("period_end", RES))
    .groupBy(*RUN_KEY, "period_end").agg(*[F.avg(c).alias(c) for c in IRR_POWER + WEATHER])
)
fc15 = write(fc15, "fe_reuni_da_wa_15min", partition_by=("horizon",))

target15 = (
    st.withColumn("period_end", ceil_to("period_end", RES))
    .groupBy("site", "period_end").agg(
        F.avg("target_mw").alias("label_target_mw"), F.avg("actual_mw").alias("label_actual_mw"),
        F.count("*").alias("_n"), F.min(F.col("label_ok").cast("int")).alias("_ok"),
        F.first("capacity_mw").alias("capacity_mw"))
    .withColumn("label_ok", (F.col("_n") == RES // STEP_MIN) & (F.col("_ok") == 1))
    .drop("_n", "_ok")
)
display(target15.groupBy("site").agg(F.count("*").alias("periods"), F.round(100 * F.avg(F.col("label_ok").cast("int")), 1).alias("label_ok_pct")))

✅ **What you should see:** 96 periods per day per site; `label_ok_pct` high (~90%+). It's lower for pv2 before Jun 2023, when `Pv2_AVL` didn't exist yet, but training starts in 2024.

## Step 5 · Origins and targets
Two origins a day (09:00 and 15:00 Gulf = 05:00 and 11:00 UTC), each predicting the next 24 h at 15-min steps.

In [ ]:
sites = spark.createDataFrame([(s,) for s in SITES], "site string")
days = spark.range(1).select(F.explode(F.sequence(F.to_date(F.lit(START)), F.to_date(F.lit(END)))).alias("d"))
origins = days.select(F.explode(F.array(*[F.lit(h) for h in ORIGIN_HOURS_GULF])).alias("h"), "d").select(
    F.timestamp_seconds(F.unix_timestamp(F.col("d").cast("timestamp")) + (F.col("h") - GULF_OFFSET_H) * 3600).alias("origin")
).crossJoin(sites)
rows = origins.select("site", "origin", F.explode(F.sequence(
    shift_minutes("origin", RES), shift_minutes("origin", WINDOW_MIN), F.expr(f"INTERVAL {RES} MINUTES"))).alias("period_end"))
print("origins:", origins.count(), "| rows:", rows.count())

✅ **What you should see:** origins = days × 2 × 2 sites; rows = origins × 96.

## Step 6 · Reuniwatt values from the latest run available at each origin
For each origin, the latest run issued **≥ 15 min before** it: day-ahead (max age 6 h) and week-ahead (max age 24 h). Its 15-min values are attached as `da_*` / `wa_*`. Weather (`temp_c`, `humidity_pct`, `dewpoint_c`, `cloud_opacity`, `clearness_index`, `pressure_hpa`, `precip_rate_mmh`) exists only in day-ahead runs. `f_*` = day-ahead, falling back to week-ahead.

In [ ]:
for h, p in PREFIX.items():
    runs = fc15.where(F.col("horizon") == h).select("site", "reference_time").distinct()
    a = asof_runs(origins, runs, LATENCY_MIN).select(
        "site", "origin", F.when(F.col("run_age_min") <= MAX_RUN_AGE[h], F.col("run_time")).alias(f"{p}_run_time"),
        F.col("run_age_min").alias(f"{p}_run_age_min"))
    cols = IRR_POWER + (WEATHER if h == "dayahead" else [])
    vals = fc15.where(F.col("horizon") == h).select("site", F.col("reference_time").alias(f"{p}_run_time"), "period_end",
                                                   *[F.col(c).alias(f"{p}_{c}") for c in cols])
    rows = rows.join(a, ["site", "origin"], "left").join(vals, ["site", f"{p}_run_time", "period_end"], "left")
for c in IRR_POWER:
    rows = rows.withColumn(f"f_{c}", F.coalesce(f"da_{c}", f"wa_{c}"))
rows = write(rows, "fe_dayahead_15min_v1_stage_a", partition_by=("site",))
display(rows.agg(*[F.round(100 * F.avg(F.col(c).isNotNull().cast("int")), 1).alias(f"{c}_pct")
                   for c in ["da_run_time", "wa_run_time", "f_ghi", "f_gti_poa", "f_power_mw", "da_temp_c"]]))

✅ **What you should see:** `da_run_time`, `f_ghi`, `f_gti_poa`, `f_power_mw` near **100%** (gaps = days Reuniwatt sent nothing or runs were broken). `da_temp_c` close to `da_run_time`.

## Step 7 · Computed and Reuniwatt-derived features (at the target)
| Feature | Meaning |
|---|---|
| `sun_zenith`, `sun_elevation`, `cos_zenith`, `air_mass`, `etr_horizontal`, `clearsky_ghi`, `is_day` | Sun at the target (computed) |
| `t_hour_sin/cos`, `t_doy_sin/cos`, `o_hour_sin/cos`, `lead_min` | Calendar (Gulf time) and lead |
| `csi_ghi`, `kt` | Forecast GHI ÷ clear-sky / ÷ top-of-atmosphere |
| `diffuse_frac`, `beam_frac` | DHI ÷ GHI (haze), BHI ÷ GHI |
| `power_norm`, `band_width_norm`, `clip_flag` | Reuniwatt power ÷ capacity, (p90 − p10) ÷ capacity, near the plant limit |
| `t_cell_c`, `temp_derate` | Panel temperature from forecast temperature and GTI; heat loss |
| `disagree_da_wa` | Day-ahead vs week-ahead power difference ÷ capacity |

In [ ]:
cap = site_value("capacity_mw")
rows = add_solar(rows, "period_end", period_min=RES)
rows = add_calendar(rows, "period_end", "t_")
rows = add_calendar(rows, "origin", "o_").drop("o_doy_sin", "o_doy_cos")
rows = (
    rows.withColumn("capacity_mw", cap)
    .withColumn("lead_min", (F.unix_timestamp("period_end") - F.unix_timestamp("origin")) / 60)
    .withColumn("csi_ghi", F.when(F.col("clearsky_ghi") > 50, F.least(F.col("f_ghi") / F.col("clearsky_ghi"), F.lit(1.5))))
    .withColumn("kt", F.when(F.col("etr_horizontal") > 50, F.col("f_ghi") / F.col("etr_horizontal")))
    .withColumn("diffuse_frac", F.when(F.col("f_ghi") > 20, F.least(F.col("f_dhi") / F.col("f_ghi"), F.lit(1.0))))
    .withColumn("beam_frac", F.when(F.col("f_ghi") > 20, F.least(F.col("f_bhi") / F.col("f_ghi"), F.lit(1.0))))
    .withColumn("power_norm", F.col("f_power_mw") / cap)
    .withColumn("band_width_norm", (F.col("f_power_p90_mw") - F.col("f_power_p10_mw")) / cap)
    .withColumn("clip_flag", (F.col("f_power_mw") >= 0.97 * cap).cast("int"))
    .withColumn("t_cell_c", F.col("da_temp_c") + 0.031 * F.col("f_gti_poa"))
    .withColumn("temp_derate", 1 - 0.004 * (F.col("t_cell_c") - 25))
    .withColumn("disagree_da_wa", F.abs(F.col("da_power_mw") - F.col("wa_power_mw")) / cap)
)
display(rows.where("is_day").select("csi_ghi", "diffuse_frac", "power_norm", "band_width_norm", "t_cell_c", "temp_derate", "disagree_da_wa")
        .summary("min", "25%", "50%", "75%", "max"))

✅ **What you should see (daytime):** `csi_ghi` mostly 0.6–1.1 · `diffuse_frac` ~0.1–0.4 · `power_norm` 0–~1 · `t_cell_c` ~25–75 °C · `temp_derate` ~0.8–1.0 · `disagree_da_wa` mostly near 0.

## Step 8 · Sensor features at the origin (`s_`)
From the plant sensors and the target, using only data **up to the origin**:

| Feature | Meaning |
|---|---|
| `s_gti_now`, `s_ghi_now`, `s_target_norm_now` | Latest value (target ÷ capacity) |
| `s_<x>_mean_15m/30m/60m`, `s_<x>_std_60m` | Average and variability over the last 15 / 30 / 60 min (GTI, GHI, target) |
| `s_<x>_trend_30m` | Change over the last 30 min |
| `s_temp_now`, `s_temp_mean_60m`, `s_humidity_now`, `s_humidity_mean_60m` | Weather now |
| `s_csi_now`, `s_csi_60m` | Measured clear-sky index (measured GHI ÷ clear-sky): how clear or hazy it is now |
| `s_yday_csi`, `s_yday_target_peak_norm` | Yesterday's (Gulf day) clear-sky index and peak target ÷ capacity: haze and availability persist |

In [ ]:
ts = F.unix_timestamp("period_end")
w = Window.partitionBy("site").orderBy(ts)


def win(minutes):
    return w.rangeBetween(-(minutes - STEP_MIN) * 60, 0)


ser = st.select("site", "period_end", "m_gti_poa", "m_ghi", "m_temp_c", "m_humidity_pct", "clearsky_ghi", "capacity_mw",
                F.when(F.col("label_ok"), F.col("target_mw") / F.col("capacity_mw")).alias("target_norm"))
SRC = {"gti": "m_gti_poa", "ghi": "m_ghi", "target_norm": "target_norm"}
sf = ser
for name, col in SRC.items():
    sf = sf.withColumn(f"s_{name}_now", F.col(col))
    for m in (15, 30, 60):
        sf = sf.withColumn(f"s_{name}_mean_{m}m", F.avg(col).over(win(m)))
    sf = sf.withColumn(f"s_{name}_std_60m", F.stddev(col).over(win(60)))
    sf = sf.withColumn(f"s_{name}_trend_30m", F.col(col) - F.lag(col, 30 // STEP_MIN).over(w))
sf = (
    sf.withColumn("s_temp_now", F.col("m_temp_c")).withColumn("s_temp_mean_60m", F.avg("m_temp_c").over(win(60)))
    .withColumn("s_humidity_now", F.col("m_humidity_pct")).withColumn("s_humidity_mean_60m", F.avg("m_humidity_pct").over(win(60)))
    .withColumn("s_csi_now", F.when(F.col("clearsky_ghi") > 50, F.least(F.col("m_ghi") / F.col("clearsky_ghi"), F.lit(1.5))))
    .withColumn("_g60", F.sum(F.when(F.col("clearsky_ghi") > 50, F.col("m_ghi"))).over(win(60)))
    .withColumn("_c60", F.sum(F.when(F.col("clearsky_ghi") > 50, F.col("clearsky_ghi"))).over(win(60)))
    .withColumn("s_csi_60m", F.when(F.col("_c60") > 0, F.col("_g60") / F.col("_c60")))
)
S_COLS = [c for c in sf.columns if c.startswith("s_")]
s_at_origin = sf.select("site", F.col("period_end").alias("origin"), *S_COLS)

daily = (
    st.where("clearsky_ghi > 50").groupBy("site", F.to_date(utc_to_gulf("period_end")).alias("gulf_day"))
    .agg((F.sum("m_ghi") / F.sum("clearsky_ghi")).alias("s_yday_csi"),
         (F.max(F.when(F.col("label_ok"), F.col("target_mw"))) / F.first("capacity_mw")).alias("s_yday_target_peak_norm"))
    .select("site", F.date_add("gulf_day", 1).alias("origin_gulf_day"), "s_yday_csi", "s_yday_target_peak_norm")  # yesterday's value, keyed by today
)
rows = (
    rows.join(s_at_origin, ["site", "origin"], "left")
    .withColumn("origin_gulf_day", F.to_date(utc_to_gulf("origin")))
    .join(daily, ["site", "origin_gulf_day"], "left")
    .drop("origin_gulf_day")
)
display(s_at_origin.join(origins, ["site", "origin"]).select(*S_COLS).summary("count", "min", "50%", "max"))

✅ **What you should see:** at the 09:00 and 15:00 Gulf origins the sun is up, so `s_gti_now` and `s_target_norm_now` > 0 and `s_csi_now` ~0.6–1.0. Counts lower for `s_gti_*` (PV2 panel sensor only from 2026).

## Step 9 · Combined features (`c_`): Reuniwatt's recent error
For each past 5-min period, take Reuniwatt's latest day-ahead forecast issued before it, and compare it with what happened:

| Feature | Meaning |
|---|---|
| `c_err_power_60m`, `c_err_power_24h` | Actual target ÷ Reuniwatt power, over the last hour / day (1 = spot on, 0.9 = Reuniwatt 10% too high) |
| `c_err_gti_60m`, `c_err_gti_24h` | Measured panel sunlight ÷ forecast panel sunlight |
| `c_csi_gap_now` | Measured clear-sky index now − Reuniwatt's at the same moment (haze Reuniwatt is missing) |
| `c_power_adj_24h`, `c_gti_adj_24h` | Reuniwatt power / GTI for the target × last-24 h error ratio (clipped 0.5–1.5) |

In [ ]:
nowcast = (
    fc5.join(br.select(*RUN_KEY, "run_broken"), RUN_KEY).where("horizon = 'dayahead' AND NOT run_broken")
    .withColumn("_rk", F.row_number().over(Window.partitionBy("site", "period_end").orderBy(F.desc("reference_time"))))
    .where("_rk = 1").select("site", "period_end", F.col("power_mw").alias("nc_power"), F.col("gti_poa").alias("nc_gti"), F.col("ghi").alias("nc_ghi"))
)
cs = st.select("site", "period_end", "clearsky_ghi", "m_gti_poa", "m_ghi",
               F.when(F.col("label_ok"), F.col("target_mw")).alias("tgt")).join(nowcast, ["site", "period_end"], "left")
day_ok = F.col("clearsky_ghi") > 50


def ratio(num, den, minutes):
    both = day_ok & F.col(num).isNotNull() & F.col(den).isNotNull()
    n = F.sum(F.when(both, F.col(num))).over(win(minutes))
    d = F.sum(F.when(both, F.col(den))).over(win(minutes))
    return F.when(d > 1, n / d)


comb = (
    cs.withColumn("c_err_power_60m", ratio("tgt", "nc_power", 60))
    .withColumn("c_err_power_24h", ratio("tgt", "nc_power", 1440))
    .withColumn("c_err_gti_60m", ratio("m_gti_poa", "nc_gti", 60))
    .withColumn("c_err_gti_24h", ratio("m_gti_poa", "nc_gti", 1440))
    .withColumn("c_csi_gap_now", F.when(day_ok, (F.col("m_ghi") - F.col("nc_ghi")) / F.col("clearsky_ghi")))
    .select("site", F.col("period_end").alias("origin"), "c_err_power_60m", "c_err_power_24h", "c_err_gti_60m", "c_err_gti_24h", "c_csi_gap_now")
)
rows = rows.join(comb, ["site", "origin"], "left")
rows = (
    rows.withColumn("c_power_adj_24h", F.col("f_power_mw") * F.least(F.greatest(F.col("c_err_power_24h"), F.lit(0.5)), F.lit(1.5)))
    .withColumn("c_gti_adj_24h", F.col("f_gti_poa") * F.least(F.greatest(F.col("c_err_gti_24h"), F.lit(0.5)), F.lit(1.5)))
)
display(comb.join(origins, ["site", "origin"]).groupBy("site").agg(
    *[F.round(F.percentile_approx(c, 0.5), 3).alias(f"{c}_median") for c in ["c_err_power_60m", "c_err_power_24h", "c_err_gti_24h", "c_csi_gap_now"]]))

✅ **What you should see:** medians near **1** for the `c_err_*` ratios (pv2 power maybe slightly above 1, since available power > Reuniwatt's forecast of actual; GTI ratios below 1 for pv2, since Reuniwatt overestimates PV2 GTI) and `c_csi_gap_now` near 0 (slightly negative = hazier than forecast).

## Step 10 · Add the label and write `fe_dayahead_15min_v1`

In [ ]:
rows = rows.join(target15.drop("capacity_mw"), ["site", "period_end"], "left") \
    .withColumn("label_target_norm", F.col("label_target_mw") / F.col("capacity_mw")) \
    .withColumn("label_ok", F.coalesce("label_ok", F.lit(False))) \
    .withColumn("product", F.lit("dayahead_15min"))
out = write(rows.where(F.col("da_run_time").isNotNull() | F.col("wa_run_time").isNotNull()), "fe_dayahead_15min_v1", partition_by=("site",))
FEATURES = [c for c in out.columns if c.startswith(("da_", "wa_", "f_", "s_", "c_")) and not c.endswith(("_run_time",))] + [
    "sun_zenith", "sun_elevation", "cos_zenith", "air_mass", "etr_horizontal", "clearsky_ghi", "is_day",
    "t_hour_sin", "t_hour_cos", "t_doy_sin", "t_doy_cos", "o_hour_sin", "o_hour_cos", "lead_min",
    "csi_ghi", "kt", "diffuse_frac", "beam_frac", "power_norm", "band_width_norm", "clip_flag", "t_cell_c", "temp_derate", "disagree_da_wa"]
print(f"{len(FEATURES)} features")

## Step 11 · Final checks and the Reuniwatt baseline
Baseline = Reuniwatt's own day-ahead power forecast, on usable daytime rows. Our model has to beat it.
- `nmae_pct`: average error ÷ capacity.
- `mape_pct`: average % error, only where the target is above 10% of capacity (avoids dividing by tiny numbers).

In [ ]:
day = out.where("is_day AND label_ok AND f_power_mw IS NOT NULL")
baseline = day.groupBy("site").agg(
    F.count("*").alias("rows"), F.countDistinct("origin").alias("origins"),
    F.round(F.corr("f_power_mw", "label_target_mw"), 4).alias("corr"),
    F.round(100 * F.avg(F.abs(F.col("f_power_mw") - F.col("label_target_mw"))) / F.first("capacity_mw"), 2).alias("nmae_pct"),
    F.round(100 * F.avg(F.when(F.col("label_target_mw") > 0.1 * F.col("capacity_mw"),
                               F.abs(F.col("f_power_mw") - F.col("label_target_mw")) / F.col("label_target_mw"))), 2).alias("mape_pct"),
    F.round(100 * F.avg(F.when(F.col("label_target_mw") > 0.1 * F.col("capacity_mw"),
                               F.abs(F.col("c_power_adj_24h") - F.col("label_target_mw")) / F.col("label_target_mw"))), 2).alias("mape_pct_adj24h"),
)
display(baseline)
nulls = out.select(*[F.round(100 * F.avg(F.col(c).isNull().cast("int")), 1).alias(c) for c in FEATURES])
display(nulls)

✅ **What you should see:**
- `corr` ~0.98.
- `mape_pct` is the **number to beat**.
- `mape_pct_adj24h` is the same forecast corrected by yesterday's error. If it's already lower, the combined features are useful.
- Null %: low for `da_`/`f_`/computed; higher for `wa_` (fallback only), weather on days without a day-ahead run, and PV2 `s_gti_*` before 2026.

## Step 12 · Write the summary file
`results/fe_dayahead_15min_v1.txt` next to this notebook. Commit and push it to share.

In [ ]:
import os
from datetime import datetime


def as_text(df, title, max_rows=100):
    rs = df.limit(max_rows).collect()
    f = lambda v: "" if v is None else (f"{v:.4g}" if isinstance(v, float) else str(v))
    return "\n".join([f"### {title}", " | ".join(df.columns)] + [" | ".join(f(v) for v in r) for r in rs]) + "\n"


nulls_long = nulls.select(F.explode(F.array(*[F.struct(F.lit(c).alias("feature"), F.col(c).alias("null_pct")) for c in FEATURES])).alias("x")).select("x.*")
text = "\n".join([
    f"# fe_dayahead_15min_v1 · origins {START} → {END} · run {datetime.utcnow():%Y-%m-%d %H:%M} UTC",
    f"table: {out_table('fe_dayahead_15min_v1')} · rows {out.count():,} · features {len(FEATURES)}\n",
    as_text(br.groupBy("site", "horizon").agg(F.count("*").alias("runs"), F.sum(F.col("run_broken").cast("int")).alias("broken")), "broken runs"),
    as_text(baseline, "Reuniwatt day-ahead baseline (daytime, label_ok)"),
    as_text(nulls_long, "null % per feature", 200),
])
os.makedirs("results", exist_ok=True)
with open("results/fe_dayahead_15min_v1.txt", "w") as fh:
    fh.write(text)
print(text)